# 📄 On-Device PDF Analyzer with Local LLM
**Powered by Ollama + LangChain**

This notebook lets you analyze any PDF(s) using a local LLM — no internet, no API keys, fully private.

### Quick Setup
1. Install [Ollama](https://ollama.com) and run `ollama pull llama3`
2. Drop your PDFs into the `pdfs/` folder
3. Run cells top to bottom

In [ ]:
# ─────────────────────────────────────────────
# Cell 1: Install & Import Dependencies
# ─────────────────────────────────────────────
# Uncomment the line below if running for the first time:
# !pip install langchain-community pypdf langchain-ollama

import os
import pypdf
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("✅ Libraries imported successfully.")

In [ ]:
# ─────────────────────────────────────────────
# Cell 2: Configuration — Edit These Settings
# ─────────────────────────────────────────────

# Folder containing your PDF files
PDF_FOLDER = "pdfs"

# Ollama model to use (must be pulled first via: ollama pull <model>)
# Options: "llama3:8b", "mistral", "gemma:7b", "phi3", etc.
MODEL_NAME = "llama3:8b"

# Max characters sent to LLM per document (prevents context overflow)
MAX_CHARS = 12000

# Output video filename
OUTPUT_VIDEO = "analysis_output.mp4"

print(f"📁 PDF Folder : {PDF_FOLDER}")
print(f"🤖 Model      : {MODEL_NAME}")
print(f"📝 Max chars  : {MAX_CHARS:,}")

In [ ]:
# ─────────────────────────────────────────────
# Cell 3: Load PDFs from Folder
# ─────────────────────────────────────────────

def load_pdfs(folder_path):
    """Load all PDFs from a folder and return list of {filename, text} dicts."""
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(
            f"PDF folder not found: '{folder_path}'\n"
            f"Please create the folder and add your PDFs."
        )

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

    if not pdf_files:
        raise ValueError(f"No PDF files found in '{folder_path}'.")

    print(f"Found {len(pdf_files)} PDF(s) in '{folder_path}':\n")
    loaded = []

    for pdf_file in sorted(pdf_files):
        pdf_path = os.path.join(folder_path, pdf_file)
        try:
            reader = pypdf.PdfReader(pdf_path)
            pages_text = []
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    pages_text.append(text.strip())

            full_text = "\n".join(pages_text)
            num_pages = len(reader.pages)
            num_chars = len(full_text)

            print(f"  ✅ {pdf_file}")
            print(f"     Pages: {num_pages} | Characters extracted: {num_chars:,}")

            if num_chars == 0:
                print(f"     ⚠️  Warning: No text extracted. PDF may be image-based (scanned).")

            loaded.append({
                "filename": pdf_file,
                "text": full_text,
                "pages": num_pages
            })

        except Exception as e:
            print(f"  ❌ {pdf_file} — Error: {e}")

    print(f"\n📦 Total PDFs successfully loaded: {len(loaded)}")
    return loaded


documents = load_pdfs(PDF_FOLDER)

# Preview first 500 chars of first document
if documents:
    print("\n" + "─" * 50)
    print(f"Preview — {documents[0]['filename']}:")
    print("─" * 50)
    print(documents[0]["text"][:500] + "...")

In [ ]:
# ─────────────────────────────────────────────
# Cell 4: Connect to Local LLM via Ollama
# ─────────────────────────────────────────────

def connect_llm(model_name):
    """Initialize and test the Ollama LLM connection."""
    print(f"Connecting to local model: {model_name}...")
    try:
        llm = ChatOllama(model=model_name)
        # Quick ping to verify connection
        test = llm.invoke("Reply with only the word: ready")
        print(f"✅ Connected! Model responded: '{test.content.strip()}'")
        return llm
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        print("\nTroubleshooting:")
        print("  1. Make sure Ollama is running: open a terminal and run 'ollama serve'")
        print(f"  2. Make sure the model is pulled: 'ollama pull {model_name}'")
        raise


llm = connect_llm(MODEL_NAME)

In [ ]:
# ─────────────────────────────────────────────
# Cell 5: Build the Analysis Prompt & Chain
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """
You are an expert academic assistant helping a college student understand and study documents.
Analyze the provided document text and produce a clear, structured report in Markdown format.
Be concise but thorough. Use headers, bullet points, and bold text appropriately.
""".strip()

ANALYSIS_TEMPLATE = """
Here is the document text to analyze:
---
{document_text}
---

Please provide the following analysis:

1. **Summary** — A concise 2-3 sentence overview of the document's main argument or purpose.

2. **Key Takeaways** — A bulleted list of the most important concepts, findings, or ideas.

3. **Important Keywords** — List the 5-10 most critical terms or concepts. Briefly define each.

4. **Resources Mentioned** — Any books, papers, websites, or tools referenced. If none, state "None found."

5. **Most Memorable Quote** — The single most impactful or representative quote from the document.

6. **Study Questions** — Generate 3 thoughtful questions a student should be able to answer after reading this.

7. **ELI5 (Explain Like I'm 5)** — Explain the core idea of this document as simply as possible.

8. **Potential Exam Topics** — What topics from this document are most likely to appear on a test or quiz?
""".strip()

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", ANALYSIS_TEMPLATE),
])

chain = prompt | llm | StrOutputParser()

print("✅ Analysis chain ready.")

In [ ]:
# ─────────────────────────────────────────────
# Cell 6: Run Analysis on All Loaded PDFs
# ─────────────────────────────────────────────

def truncate_text(text, max_chars):
    """Truncate text to fit within LLM context window."""
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars]
    print(f"  ⚠️  Text truncated to {max_chars:,} characters to fit model context.")
    return truncated + "\n\n[... document truncated for length ...]"


def analyze_documents(documents, chain, max_chars):
    """Run the LLM analysis chain on each loaded document."""
    all_results = []

    for i, doc in enumerate(documents, 1):
        print(f"\n{'═' * 55}")
        print(f"📄 Analyzing document {i}/{len(documents)}: {doc['filename']}")
        print(f"{'═' * 55}")

        text = truncate_text(doc["text"], max_chars)

        if not text.strip():
            print("  ❌ Skipping — no text content found.")
            continue

        try:
            print("  🤖 Running analysis (this may take 30–90 seconds)...\n")
            response = chain.invoke({"document_text": text})

            result = {
                "filename": doc["filename"],
                "analysis": response
            }
            all_results.append(result)

            print(response)

        except Exception as e:
            print(f"  ❌ Analysis failed for {doc['filename']}: {e}")

    return all_results


results = analyze_documents(documents, chain, MAX_CHARS)
print(f"\n✅ Analysis complete. Processed {len(results)}/{len(documents)} document(s).")

In [ ]:
# ─────────────────────────────────────────────
# Cell 7: Save Analysis to Markdown File
# ─────────────────────────────────────────────

def save_analysis(results, output_path="analysis_report.md"):
    """Save all analysis results to a single Markdown file."""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("# PDF Analysis Report\n\n")
        f.write(f"**Documents analyzed:** {len(results)}\n\n")
        f.write("---\n\n")

        for result in results:
            f.write(f"## 📄 {result['filename']}\n\n")
            f.write(result['analysis'])
            f.write("\n\n---\n\n")

    print(f"✅ Analysis saved to: {output_path}")


if results:
    save_analysis(results, "analysis_report.md")

In [ ]:
# ─────────────────────────────────────────────
# Cell 8: (Optional) Generate Scrolling MP4 Video
# ─────────────────────────────────────────────
# Requires: ffmpeg installed on your system
# Install: https://ffmpeg.org/download.html

import matplotlib.pyplot as plt
from matplotlib import animation
import textwrap

def create_scrolling_mp4(
    text,
    out_file="analysis_output.mp4",
    fps=20,
    lines_in_window=25,
    wrap_width=90,
    fontsize=11,
    figsize=(13, 7),
    bg_color="#1e1e2e",
    text_color="#cdd6f4",
    bitrate=2000
):
    """
    Generate a smooth scrolling MP4 video of the analysis text.
    Uses a dark theme for readability.
    """
    plt.rcParams["font.family"] = "DejaVu Sans"

    # Wrap each paragraph
    lines = []
    for paragraph in text.split("\n"):
        if paragraph.strip() == "":
            lines.append("")
        else:
            lines.extend(textwrap.wrap(paragraph, width=wrap_width) or [""])

    fig, ax = plt.subplots(figsize=figsize, facecolor=bg_color)
    ax.set_facecolor(bg_color)
    ax.axis("off")

    line_height = 1.0
    text_objs = []

    for i, line in enumerate(lines):
        y = -i * line_height
        t = ax.text(
            0.02, y, line,
            va="top", ha="left",
            fontsize=fontsize,
            color=text_color,
            transform=ax.transData
        )
        text_objs.append(t)

    top = 0.5
    bottom = top - lines_in_window
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom, top)

    frames_per_line = max(1, int(fps * 0.15))
    total_scroll = max(1, len(lines) - lines_in_window)
    total_frames = total_scroll * frames_per_line

    def update(frame):
        shift = (frame / frames_per_line) * line_height
        new_top = top - shift
        ax.set_ylim(new_top - lines_in_window, new_top)
        return text_objs

    ani = animation.FuncAnimation(fig, update, frames=total_frames, blit=False)
    writer = animation.FFMpegWriter(fps=fps, bitrate=bitrate)

    print(f"🎬 Rendering {total_frames} frames to {out_file}...")
    ani.save(out_file, writer=writer)
    plt.close(fig)
    print(f"✅ MP4 saved to: {out_file}")


# Combine all analysis into one video
if results:
    combined_text = "\n\n".join(
        f"{'='*50}\n{r['filename']}\n{'='*50}\n\n{r['analysis']}"
        for r in results
    )
    create_scrolling_mp4(combined_text, out_file=OUTPUT_VIDEO)
else:
    print("No results to render. Run Cell 6 first.")